# 🌸 Nhận diện hoa với CNN và PyTorch

Bài thực hành này giúp bạn xây dựng mô hình học sâu sử dụng mạng nơ-ron tích chập (CNN) để phân loại hình ảnh các loài hoa từ bộ dữ liệu **Flowers Recognition** (Kaggle).

In [1]:
!pip install -q gradio
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import os
import kagglehub
from PIL import Image
import gradio as gr

## 🔽 Load dữ liệu

In [2]:
# Download latest version
path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'flowers-recognition' dataset.
Path to dataset files: /kaggle/input/flowers-recognition


## ⚙️ Tiền xử lý dữ liệu và tạo DataLoader

In [3]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = datasets.ImageFolder("/kaggle/input/flowers-recognition/flowers", transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = dataset.classes
print("Các lớp:", classes)

Các lớp: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']


## 🧠 Xây dựng mô hình CNN

In [4]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes):
        super(FlowerCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = FlowerCNN(num_classes=len(classes))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

FlowerCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=5, bias=True)
  )
)

## 🏋️ Huấn luyện mô hình

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    running_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.2137
Epoch 2, Loss: 0.9196
Epoch 3, Loss: 0.7267
Epoch 4, Loss: 0.5287
Epoch 5, Loss: 0.3215
Epoch 6, Loss: 0.1357
Epoch 7, Loss: 0.0868
Epoch 8, Loss: 0.0528
Epoch 9, Loss: 0.0264
Epoch 10, Loss: 0.0266


## 🎯 Đánh giá mô hình

In [6]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Độ chính xác: {100 * correct / total:.2f}%")

Độ chính xác: 61.57%


## 💾 Lưu và tải mô hình

In [7]:
# Lưu
torch.save(model.state_dict(), "flower_cnn.pth")

# Tải lại
model = FlowerCNN(num_classes=len(classes))
model.load_state_dict(torch.load("flower_cnn.pth"))
model.to(device)
model.eval()

FlowerCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=5, bias=True)
  )
)

## 🌐 Giao diện dự đoán với Gradio

In [8]:
def predict(img):
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img)
        _, pred = torch.max(output, 1)
    return classes[pred.item()]

gr.Interface(fn=predict, inputs=gr.Image(type="pil"), outputs="label").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3b7401d10a91e30957.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


##Thực hành mở rộng:
1. Sử dụng mô hình học sâu khác (ResNet, MobileNet, v.v.)
2. Thử nghiệm với bộ dữ liệu khác như Oxford 102 Flower hoặc Plant Seedlings
3. Cải thiện mô hình bằng các kỹ thuật học sâu (augmentation, dropout, learning rate scheduling, ...)